In [2]:
"""
Asymmetric distillation factory search (Section IV).

Search over (l, n, k, s_total, s_O) parameter tuples, with s_S = 1 fixed.
For each tuple, the gate-type set is fully determined by the skip parameters;
the only freedom is the sign assignment on w_O = 0 gates.

Verifies the borrowed-identity condition: Phi(i_O, i_S) ≡ Phi(0, 0) (mod 2pi)
for all (i_O, i_S) — equivalently, Phi is constant.

Output extraction removes all (w, 0) gates with w ∈ W_O ∩ W_total.
The factory is [[N_circuit - sum_w C(k, w), k, d]] where d = 2 for k ≥ 2,
or d = s_total + 1 for k = 1.
"""

from math import comb
from itertools import product
import time


# --- Phase polynomial --------------------------------------------------------

def f_coefficient(i_O, i_S, k, n_minus_k, w_O, w_S):
    """Coefficient [y^w_O z^w_S] of f_{i_O, i_S}(y, z)."""
    coeff_y = sum(
        ((-1) ** j) * comb(i_O, j) * comb(k - i_O, w_O - j)
        for j in range(max(0, w_O - (k - i_O)), min(i_O, w_O) + 1)
    )
    coeff_z = sum(
        ((-1) ** j) * comb(i_S, j) * comb(n_minus_k - i_S, w_S - j)
        for j in range(max(0, w_S - (n_minus_k - i_S)), min(i_S, w_S) + 1)
    )
    return coeff_y * coeff_z


def phi_over_theta_int(i_O, i_S, k, n_minus_k, sigma_dict):
    """Compute Phi(i_O, i_S) / theta as integer (Lemma 7)."""
    total = 0
    for (w_O, w_S), sigma in sigma_dict.items():
        if sigma == 0:
            continue
        c = f_coefficient(i_O, i_S, k, n_minus_k, w_O, w_S)
        total -= sigma * c
    return total


# --- Borrowed-identity condition ---------------------------------------------
'''
def is_borrowed_identity(l, n, k, sigma_dict):
    """
    Theorem 8: Phi(i_O, i_S) ≡ Phi(0, 0) (mod 2^(l+1)) for all (i_O, i_S).
    """
    n_minus_k = n - k
    full_mod = 1 << (l + 1)
    phi_baseline = phi_over_theta_int(0, 0, k, n_minus_k, sigma_dict) % full_mod
    for i_O in range(k + 1):
        for i_S in range(n_minus_k + 1):
            phi = phi_over_theta_int(i_O, i_S, k, n_minus_k, sigma_dict) % full_mod
            if phi != phi_baseline:
                return False
    return True
'''
def is_borrowed_identity(l, n, k, sigma_dict):
    n_minus_k = n - k
    for i_O in range(k + 1):
        for i_S in range(n_minus_k + 1):
            if i_O + i_S == 0 or i_O + i_S > l:
                continue
            required_div = 1 << (l - i_O - i_S + 1)  # 2^(l - i_O - i_S + 1)
            total = sum(
                sigma * comb(k - i_O, w_O - i_O) * comb(n_minus_k - i_S, w_S - i_S)
                for (w_O, w_S), sigma in sigma_dict.items()
                if w_O >= i_O and w_S >= i_S
            )
            if total % required_div != 0:
                return False
    return True

# --- Output extraction and factory parameters --------------------------------

def n_circuit_count(k, n_minus_k, sigma_dict):
    """Total gates with sigma != 0."""
    return sum(
        comb(k, w_O) * comb(n_minus_k, w_S)
        for (w_O, w_S), sigma in sigma_dict.items()
        if sigma != 0
    )


def extract_factory(k, n_minus_k, sigma_dict, W_O, W_total):
    """
    Compute the residual phase polynomial after extracting (w_O, 0) gates
    with w_O ∈ W_O ∩ W_total.

    Returns:
        (N_paper, residual_phase_dict)
    where residual_phase_dict[i_O] = (Phi'(i_O, 0) - Phi'(0, 0)) / theta mod 2^(l+1)
    """
    N_circuit = n_circuit_count(k, n_minus_k, sigma_dict)
    intersection = set(W_O) & set(W_total)
    n_removed = sum(comb(k, w) for w in intersection if w >= 1)
    N_paper = N_circuit - n_removed
    return N_paper


def residual_phase_at_iO(l, k, n_minus_k, sigma_dict, W_O_int_W_total, i_O):
    """
    Residual Phi'(i_O, 0) after removing (w, 0) gates with w in W_O ∩ W_total.

    The (w, 0) gates contribute -sigma_{w, 0} * A_{i_O}[y^w] * B_0[z^0]
    = -sigma_{w, 0} * A_{i_O}[y^w]  (since B_0[z^0] = 1)
    to Phi(i_O, 0).
    Removing them adds +sigma_{w, 0} * A_{i_O}[y^w] back.
    """
    full_mod = 1 << (l + 1)
    phi_full = phi_over_theta_int(i_O, 0, k, n_minus_k, sigma_dict)
    # The contribution of (w, 0) gates to phi_full
    extracted = 0
    for w in W_O_int_W_total:
        if w < 1:
            continue
        sigma = sigma_dict.get((w, 0), 0)
        if sigma == 0:
            continue
        # Contribution was -sigma * A[y^w] * B[z^0] = -sigma * A[y^w]
        # Removing it adds +sigma * A[y^w]
        coeff_y = sum(
            ((-1) ** j) * comb(i_O, j) * comb(k - i_O, w - j)
            for j in range(max(0, w - (k - i_O)), min(i_O, w) + 1)
        )
        extracted -= -sigma * coeff_y  # subtract the contribution
    return (phi_full + extracted) % full_mod


def classify_output(l, k, n_minus_k, sigma_dict, W_O_int_W_total):
    """
    Compute residual phase Phi'(i_O, 0) - Phi'(0, 0) mod 2^(l+1)
    for i_O = 0, ..., k. Identify the output magic state from the
    polynomial form.
    """
    full_mod = 1 << (l + 1)
    half_mod = full_mod // 2
    
    def center(x):
        return ((x + half_mod) % full_mod) - half_mod
    
    phi_residual = [
        residual_phase_at_iO(l, k, n_minus_k, sigma_dict, W_O_int_W_total, i_O)
        for i_O in range(k + 1)
    ]
    # Subtract baseline
    phi_diff = [(p - phi_residual[0]) % full_mod for p in phi_residual]
    phi_diff = [center(p) for p in phi_diff]
    
    if all(p == 0 for p in phi_diff):
        return "trivial", phi_diff
    
    # Find lowest-order constant finite difference
    highest_d = 0
    for d in range(1, k + 1):
        c_d = sum(((-1)**(d-j)) * comb(d, j) * phi_diff[j] for j in range(d+1)) % full_mod
        c_d = center(c_d)
        if c_d != 0:
            highest_d = d
    if highest_d == 0:
        return "trivial", phi_diff
    return f"deg{highest_d}", phi_diff


# --- Skip-parameter gate-set construction ------------------------------------

def W_AP(n, s):
    """Arithmetic progression {1, 1+s, 1+2s, ...} ∩ {1, ..., n}."""
    return [w for w in range(1, n + 1) if (w - 1) % s == 0]


def build_gate_set(k, n_minus_k, s_total, s_O):
    """Return (allowed_pairs, W_total, W_O) given skip parameters with s_S = 1."""
    n = k + n_minus_k
    W_total = set(W_AP(n, s_total))
    W_O = set([0] + W_AP(k, s_O))
    W_S = set(range(0, n_minus_k + 1))  # s_S = 1

    pairs = []
    for w_O in W_O:
        if w_O > k:
            continue
        for w_S in W_S:
            if w_S > n_minus_k:
                continue
            if w_O + w_S in W_total and w_O + w_S >= 1:
                pairs.append((w_O, w_S))
    return sorted(pairs), W_total, W_O


# --- Sign-assignment search --------------------------------------------------

def find_valid_signs(l, n, k, pairs):
    """
    Try all-positive first. If it fails, enumerate σ ∈ {-1, +1} for
    w_O = 0 pairs (others fixed at +1).

    Returns: list of valid sigma_dicts.
    """
    valid_configs = []

    sigma_all_pos = {pair: 1 for pair in pairs}
    if is_borrowed_identity(l, n, k, sigma_all_pos):
        valid_configs.append(dict(sigma_all_pos))

    check_only_pairs = [p for p in pairs if p[0] == 0]
    if not check_only_pairs:
        return valid_configs

    for sign_choice in product([-1, 1], repeat=len(check_only_pairs)):
        if all(s == 1 for s in sign_choice):
            continue
        sigma_dict = {pair: 1 for pair in pairs}
        for pair, sign in zip(check_only_pairs, sign_choice):
            sigma_dict[pair] = sign
        if is_borrowed_identity(l, n, k, sigma_dict):
            valid_configs.append(dict(sigma_dict))

    return valid_configs


# --- Main search loop --------------------------------------------------------

def search_all(L_RANGE, K_RANGE, N_MAX, S_TOTAL_RANGE, S_O_RANGE, verbose=True):
    all_results = []
    timing = {}
    for l in L_RANGE:
        for k in K_RANGE:
            for n in range(k + 1, N_MAX + 1):
                n_minus_k = n - k
                for s_total in S_TOTAL_RANGE:
                    for s_O in S_O_RANGE:
                        pairs, W_total, W_O = build_gate_set(
                            k, n_minus_k, s_total, s_O
                        )
                        if not pairs:
                            continue
                        if verbose:
                            print(f"  l={l} k={k} n={n} s_total={s_total} s_O={s_O}: "
                                  f"|pairs|={len(pairs)}", flush=True)
                        t_start = time.time()
                        valid_configs = find_valid_signs(l, n, k, pairs)
                        t_elapsed = time.time() - t_start
                        timing[(l, k, n, s_total, s_O)] = {
                            "time": t_elapsed,
                            "n_pairs": len(pairs),
                            "found": len(valid_configs),
                        }
                        if verbose:
                            print(f"    → {len(valid_configs)} valid in "
                                  f"{t_elapsed:.3f}s", flush=True)
                        for sigma_dict in valid_configs:
                            N_paper = extract_factory(
                                k, n_minus_k, sigma_dict, W_O, W_total
                            )
                            # Distance: d = 2 for k >= 2; d = s_total + 1 for k = 1
                            d = (s_total + 1) if k == 1 else 2
                            W_O_int_W_total = set(W_O) & set(W_total)
                            output_type, phi_diff = classify_output(
                                l, k, n_minus_k, sigma_dict, W_O_int_W_total
                            )
                            all_results.append({
                                "l": l, "n": n, "k": k,
                                "s_total": s_total, "s_O": s_O,
                                "N": N_paper, "d": d,
                                "output": output_type,
                                "phi_diff": phi_diff,
                                "W_total": sorted(W_total),
                                "W_O": sorted(W_O),
                                "pairs": pairs,
                                "sigma": sigma_dict,
                            })
    return all_results, timing


# --- Recovery check ----------------------------------------------------------

TARGETS = [
    {"name": "[[6, 1, 2]] (Sec II all-weights, l=2)",
     "l": 2, "n": 3, "k": 1, "N": 6, "d": 2},
    {"name": "[[7, 1, 3]] (Steane / RM, l=2)",
     "l": 2, "n": 4, "k": 1, "N": 7, "d": 3},
    {"name": "[[6, 2, 2]] (H-code, l=2)",
     "l": 2, "n": 4, "k": 2, "N": 6, "d": 2},
    {"name": "[[14, 1, 2]] (Sec II all-weights, l=3)",
     "l": 3, "n": 4, "k": 1, "N": 14, "d": 2},
    {"name": "[[15, 1, 3]] (RM, l=3)",
     "l": 3, "n": 5, "k": 1, "N": 15, "d": 3},
    {"name": "[[12, 2, 2]] (Sec II derivative, l=3)",
     "l": 3, "n": 4, "k": 2, "N": 12, "d": 2},
    {"name": "[[14, 2, 2]] (BH k=2, l=3)",
     "l": 3, "n": 5, "k": 2, "N": 14, "d": 2},
    {"name": "[[8, 3, 2]] (cube, l=3)",
     "l": 3, "n": 4, "k": 3, "N": 8, "d": 2},
    {"name": "[[20, 4, 2]] (BH k=4, l=3)",
     "l": 3, "n": 7, "k": 4, "N": 20, "d": 2},
    {"name": "[[26, 6, 2]] (BH k=6, l=3, stretch)",
     "l": 3, "n": 9, "k": 6, "N": 26, "d": 2},
]


def recovery_check(results):
    print("\n" + "=" * 60)
    print("Recovery check")
    print("=" * 60)
    for t in TARGETS:
        matches = [
            r for r in results
            if r["l"] == t["l"] and r["n"] == t["n"]
            and r["k"] == t["k"] and r["N"] == t["N"] and r["d"] == t["d"]
        ]
        status = f"✓ found ({len(matches)} matches)" if matches else "✗ NOT FOUND"
        print(f"  {t['name']}: {status}")
        if matches:
            m = matches[0]
            active = [(p, m["sigma"][p]) for p in m["pairs"] if m["sigma"][p] != 0]
            print(f"    s_total={m['s_total']}, s_O={m['s_O']}, "
                  f"output={m['output']}")
            print(f"    Active gates (sigma != 0): {active}")


# --- Output ------------------------------------------------------------------

def print_summary(results):
    print(f"\nTotal valid factories found: {len(results)}")
    seen = set()
    print(f"\n{'l':<3}{'k':<4}{'N':<5}{'d':<3}{'n':<4}{'output':<10}"
          f"{'ent.':<6}{'s_t':<5}{'s_O':<5}")
    print("-" * 56)
    for r in sorted(results, key=lambda r: (r["l"], r["k"], r["d"], r["N"], r["n"])):
        key = (r["l"], r["k"], r["N"], r["d"], r["output"])
        if key in seen:
            continue
        seen.add(key)
        if r["output"] == "trivial":
            ent_label = "—"
        elif is_entangled_output(r["output"]):
            ent_label = "ent"
        else:
            ent_label = "unent"
        print(f"{r['l']:<3}{r['k']:<4}{r['N']:<5}{r['d']:<3}{r['n']:<4}"
              f"{r['output']:<10}{ent_label:<6}{r['s_total']:<5}{r['s_O']:<5}")

def is_entangled_output(output_type):
    if output_type == "trivial":
        return False
    degree = int(output_type.replace("deg", ""))
    return degree >= 2
# --- Main --------------------------------------------------------------------

if __name__ == "__main__":
    L_RANGE = [2, 3,4]
    K_RANGE = [1, 2, 3, 4, 5, 6, 7]
    N_MAX = 11
    S_TOTAL_RANGE = [1, 2, 3, 4, 5, 6, 7]
    S_O_RANGE = [1, 2, 3, 4, 5, 6, 7]

    print("=" * 60)
    print("Asymmetric distillation factory search (Section IV)")
    print(f"  l ∈ {L_RANGE}, k ∈ {K_RANGE}, n ≤ {N_MAX}")
    print(f"  s_total ∈ {S_TOTAL_RANGE}, s_O ∈ {S_O_RANGE}, s_S = 1 (fixed)")
    print("=" * 60)

    all_results, timing = search_all(
        L_RANGE, K_RANGE, N_MAX, S_TOTAL_RANGE, S_O_RANGE,
        verbose=True
    )

    print_summary(all_results)
    recovery_check(all_results)


Asymmetric distillation factory search (Section IV)
  l ∈ [2, 3, 4], k ∈ [1, 2, 3, 4, 5, 6, 7], n ≤ 11
  s_total ∈ [1, 2, 3, 4, 5, 6, 7], s_O ∈ [1, 2, 3, 4, 5, 6, 7], s_S = 1 (fixed)
  l=2 k=1 n=2 s_total=1 s_O=1: |pairs|=3
    → 0 valid in 0.000s
  l=2 k=1 n=2 s_total=1 s_O=2: |pairs|=3
    → 0 valid in 0.000s
  l=2 k=1 n=2 s_total=1 s_O=3: |pairs|=3
    → 0 valid in 0.000s
  l=2 k=1 n=2 s_total=1 s_O=4: |pairs|=3
    → 0 valid in 0.000s
  l=2 k=1 n=2 s_total=1 s_O=5: |pairs|=3
    → 0 valid in 0.000s
  l=2 k=1 n=2 s_total=1 s_O=6: |pairs|=3
    → 0 valid in 0.000s
  l=2 k=1 n=2 s_total=1 s_O=7: |pairs|=3
    → 0 valid in 0.000s
  l=2 k=1 n=2 s_total=2 s_O=1: |pairs|=2
    → 0 valid in 0.000s
  l=2 k=1 n=2 s_total=2 s_O=2: |pairs|=2
    → 0 valid in 0.000s
  l=2 k=1 n=2 s_total=2 s_O=3: |pairs|=2
    → 0 valid in 0.000s
  l=2 k=1 n=2 s_total=2 s_O=4: |pairs|=2
    → 0 valid in 0.000s
  l=2 k=1 n=2 s_total=2 s_O=5: |pairs|=2
    → 0 valid in 0.000s
  l=2 k=1 n=2 s_total=2 s_O=6: |pairs

In [3]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import numpy as np
results= all_results
MARKER_OUT = {"deg1": "o", "deg2": "s", "deg3": "^", "deg4": "D"}
COLOR_DEG  = {"deg1": "#e41a1c", "deg2": "#377eb8", "deg3": "#2ca02c", "deg4": "#ff7f00"}
LABEL_OUT  = {"deg1": r"$\mathrm{deg}=1$", "deg2": r"$\mathrm{deg}=2$", "deg3": r"$\mathrm{deg}=3$", "deg4": r"$\mathrm{deg}=4$"}
CATALYTIC  = [(10, 2, 3, "deg1"), (11, 1, 3, "deg1")]
D_JITTER   = {"deg1": -0.06, "deg2": -0.02, "deg3": 0.02, "deg4": 0.06}

# Annotations per panel — only points that actually exist and fit in N<=100
LANDMARKS = {
    2: [
        (6,  2, r"$[[6,2,2]]$",   (4,  5)),
        (6,  1, r"$[[6,1,2]]$",   (-20, 5)),
        (7,  1, r"$[[7,1,3]]$",   (4,  5)),
    ],
    3: [
        (8,  3, r"$[[8,3,2]]$",   (-35,  0)),
        (10, 2, r"$[[10,2,2]]$",  (-48,  5)),
        (11, 1, r"$[[11,1,2]]$",  (-20,  5)),
        (12, 2, r"$[[12,2,2]]$",  (-20,  5)),
        (12, 3, r"$[[12,3,2]]$",  (-55, 10)),
        (14, 1, r"$[[14,1,2]]$",  (4,   -5)),
        (14, 2, r"$[[14,2,2]]$",  (4,    10)),
        (15, 1, r"$[[15,1,3]]$",  (4, 5)),
        (20, 4, r"$[[20,4,2]]$",  (4,   5)),
        (26, 6, r"$[[26,6,2]]$",  (4,   5)),
    ],
    4: [
        (16, 4, r"$[[16,4,2]]$",  (4,   5)),
        (24, 3, r"$[[24,3,2]]$",  (-10,   5)),
        (28, 2, r"$[[28,2,2]]$",  (-30,   5)),
        (30, 1, r"$[[30,1,2]]$",  (4, 5)),
        (30, 2, r"$[[30,2,2]]$",  (4,  5)),
        (31, 1, r"$[[31,1,3]]$",  (-60, 5)),
    ],
}

seen = set()
factories = []
for r in results:
    if r["output"] == "trivial" or r["N"] > 100:
        continue
    key = (r["N"], r["k"], r["l"], r["output"])
    if key not in seen:
        seen.add(key)
        factories.append((r["N"], r["k"], r["l"], r["output"]))

fig, axes = plt.subplots(1, 3, figsize=(7.16, 2.2),
                         sharex=True, sharey=True,
                         constrained_layout=True)

k_range = np.linspace(0.7, 7, 200)

for idx, (ax, l) in enumerate(zip(axes, [2, 3, 4])):

    # gamma isolines
    for gamma in [1, 2, 3, 4, 5]:
        N_iso = k_range * (2 ** gamma)
        mask = N_iso <= 100
        ax.plot(N_iso[mask], k_range[mask], "--", color="#dddddd",
                linewidth=0.5, zorder=0)

    # Factories
    for N, k, lv, out in factories:
        if lv != l or out not in MARKER_OUT:
            continue
        jit = D_JITTER.get(out, 0)
        ax.scatter(N * (1 + jit), k, marker=MARKER_OUT[out], s=12,
                   c=[COLOR_DEG[out]], edgecolors=COLOR_DEG[out],
                   linewidths=0.3, alpha=0.85, zorder=3)

    # Catalytic open markers
    for N, k, lc, out in CATALYTIC:
        if lc != l or out not in MARKER_OUT:
            continue
        ax.scatter(N, k, marker=MARKER_OUT[out], s=20,
                   facecolors="white", edgecolors=COLOR_DEG[out],
                   linewidths=0.8, zorder=4)

    # Annotations
    for N, k, label, xytext in LANDMARKS.get(l, []):
        ax.annotate(
            label, (N, k),
            xytext=xytext, textcoords="offset points",
            fontsize=5, color="#222222", zorder=6,
            bbox=dict(boxstyle="round,pad=0.12", fc="white",
                      ec="#aaa", lw=0.3, alpha=0.9),
            arrowprops=dict(arrowstyle="-", color="#aaa",
                            lw=0.4, shrinkA=0, shrinkB=1),
        )

    ax.set_title(rf"$\ell={l}$", fontsize=7, pad=2)
    ax.set_xscale("log")
    ax.set_xlim(3, 100)
    ax.set_ylim(0.5, 7.5)
    ax.set_xlabel(r"$N$", fontsize=7)
    ax.set_yticks([1, 2, 3, 4, 5, 6, 7])
    ax.tick_params(axis="both", labelsize=6)
    ax.grid(True, linestyle=":", linewidth=0.3, alpha=0.4, zorder=0)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    # Remove y tick labels on middle and right panels
    if idx > 0:
        ax.tick_params(labelleft=False)

axes[0].set_ylabel(r"$k$", fontsize=7)

# Legend inside l=4 panel (upper left, which is mostly empty)
legend_elements = []
for deg, label in LABEL_OUT.items():
    if any(out == deg for _, _, _, out in factories):
        legend_elements.append(
            Line2D([0],[0], marker=MARKER_OUT[deg], color="w",
                   markerfacecolor=COLOR_DEG[deg], markeredgecolor=COLOR_DEG[deg],
                   markersize=4.5, label=label)
        )
legend_elements.append(
    Line2D([0],[0], marker="o", color="w",
           markerfacecolor="white", markeredgecolor=COLOR_DEG["deg1"],
           markersize=4.5, markeredgewidth=0.8, label=r"catalytic")
)
legend_elements.append(
    Line2D([0],[0], color="#bbbbbb", linestyle="--",
           linewidth=0.7, label=r"$\gamma\!=\!\log_2\frac{N}{k}$")
)

axes[2].legend(handles=legend_elements, loc="upper left",
               bbox_to_anchor=(0.03, 0.98),
               frameon=True, framealpha=0.92, edgecolor="#cccccc",
               fontsize=5.5, ncol=1, labelspacing=0.3,
               handletextpad=0.4, borderpad=0.4)

fig.savefig("code_IV.png", dpi=300, bbox_inches="tight")
print("saved")

saved


In [4]:
# Label the gamma isolines at their upper ends using a darker color for contrast.
for ax in axes:
    for gamma in [1, 2, 3, 4, 5]:
        N_iso = k_range * (2 ** gamma)
        mask = N_iso <= 100
        if not np.any(mask):
            continue
        ax.annotate(
            rf"$\gamma={gamma}$",
            (N_iso[mask][-1], k_range[mask][-1]),
            xytext=(3, 1),
            textcoords="offset points",
            color="#666666",
            fontsize=5,
            ha="left",
            va="bottom",
            zorder=5,
        )

fig.savefig("code_IV.png", dpi=300, bbox_inches="tight")
print("saved")

saved
